In [4]:
# loading environment variables and setting gemini api key

import os
from dotenv import load_dotenv
from llama_index.llms.google_genai import GoogleGenAI

load_dotenv()

my_api_key = os.environ.get("GOOGLE_API_KEY") 

llm = GoogleGenAI('gemini-2.0-flash', api_key=my_api_key)

In [ ]:
# initiating free embedding model from huggingface

from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(model_name='BAAI/bge-small-en')

/Users/vineetdorikar/Developer/projects/RAG_learnings/RAG_git/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# setting up our custom llm and embedding model as default llm and embedding model

from llama_index.core import Settings

Settings.llm = llm
Settings.embed_model = embed_model

In [ ]:
# creating two different lists of documents 

from llama_index.core import SimpleDirectoryReader

documents_1 = SimpleDirectoryReader(
    input_files=["data/multi_query_data/origin.pdf"]
).load_data()
documents_2 = SimpleDirectoryReader(
    input_files=["data/multi_query_data/survey.pdf"]
).load_data()

In [ ]:
# creating two separate indexes

from llama_index.core import VectorStoreIndex 

index_1 = VectorStoreIndex.from_documents(documents_1)
index_2 = VectorStoreIndex.from_documents(documents_2)


In [ ]:
# creating retriever object (quey fusion retriever)

from llama_index.core.retrievers import QueryFusionRetriever

retriever = QueryFusionRetriever(
    [index_1.as_retriever(), index_2.as_retriever()],
    similarity_top_k=2,
    num_queries=4,
    use_async=True,
    verbose=True
)

In [11]:
import nest_asyncio

nest_asyncio.apply()

In [12]:
nodes_with_score = retriever.retrieve("what is attention mechanism?")

Retrying llama_index.llms.google_genai.base.GoogleGenAI._chat in 0.8508489827016404 seconds as it raised ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The model is overloaded. Please try again later.', 'status': 'UNAVAILABLE'}}.


Generated queries:
attention mechanism in deep learning
attention mechanism explained simply
history of attention mechanism


In [13]:
for node in nodes_with_score:
    print(f"Score: {node.score:.2f} - {node.text[:100]}...")

Score: 0.90 - Scaled Dot-Product Attention
 Multi-Head Attention
Figure 2: (left) Scaled Dot-Product Attention. (r...
Score: 0.90 - In Proceedings of ICASSP . 5884–5888. https://doi.org/10.1109/ICASSP.2018.8462506
[32] Yihe Dong, Je...


In [14]:
from llama_index.core.query_engine import RetrieverQueryEngine

query_engine = RetrieverQueryEngine.from_args(retriever)

In [15]:
response = query_engine.query(
    "what is attention mechanism? explain."
)

Generated queries:
attention mechanism explained
attention mechanism in deep learning
how does attention mechanism work?


In [17]:
from llama_index.core.response.notebook_utils import display_response

display_response(response)

**`Final Response:`** Attention mechanisms assign weights to values by using a compatibility function between a query and its corresponding key. Two common types are additive attention and dot-product attention. Dot-product attention is faster and more space-efficient due to its implementation using optimized matrix multiplication code. However, with larger dimensions, additive attention may perform better. To address the issue of large dot product values pushing the softmax function into areas with extremely small gradients, the dot products are scaled. Multi-head attention improves upon single attention functions by linearly projecting queries, keys, and values multiple times with different, learned linear projections, and then performing the attention function in parallel.